# FYS4480/FYS9480 -- Week 41: matrix product operators and the DMRG

Companion notebook to the week-41 slides (lecture notes chapter 11, sections
11.3 and 11.5-11.7).  Everything is plain NumPy, so that each number on the
slides can be reproduced from scratch:

1. matrix product operators as automata, the operator Schmidt rank, and the
   gauge freedom of the MPO;
2. the norm matrix $\boldsymbol{G}$ of a non-canonical MPS and what
   canonicalisation does to its condition number;
3. a two-site DMRG in about 150 lines, applied to the Heisenberg chain with
   $L=32$ sites, with the sweep watched step by step;
4. numerical checks of the answers to the week-40 exercises: the stability
   matrix from Wick's theorem against a brute-force second derivative, and
   the positive-semidefiniteness tests for the Lipkin and pairing models.

In [1]:
import numpy as np
import time
np.set_printoptions(precision=6, suppress=True, linewidth=120)

## 1. Matrix product operators

An MPO is a product of $D\times D$ matrices whose entries are $d\times d$
operators.  We store $\boldsymbol{W}$ as an array of shape $(D,D,d,d)$ and
contract the product into a dense $d^L\times d^L$ matrix for small $L$,
which is only a check -- the DMRG never does this.

In [2]:
sx = np.array([[0, 1], [1, 0]]) / 2
sy = np.array([[0, -1j], [1j, 0]]) / 2
sz = np.array([[1, 0], [0, -1]]) / 2
id2 = np.eye(2)

def heisenberg_mpo(J=1.0):
    """The D=5 automaton of the nearest-neighbour Heisenberg chain."""
    W = np.zeros((5, 5, 2, 2), dtype=complex)
    W[0, 0] = id2
    W[0, 1], W[0, 2], W[0, 3] = sx, sy, sz
    W[1, 4], W[2, 4], W[3, 4] = J * sx, J * sy, J * sz
    W[4, 4] = id2
    return W

def mpo_dense(W, L, first_row=0, last_col=-1):
    """Contract W_1 W_2 ... W_L into a dense d^L x d^L matrix.
    The first site contributes row `first_row`, the last site column `last_col`."""
    D, _, d, _ = W.shape
    # running object: (D, d^k, d^k)
    M = W[first_row]                      # (D, d, d)
    for _ in range(L - 1):
        # M[b, I, J] W[b, c, s, t] -> (c, I s, J t)
        M = np.einsum('bIJ,bcst->cIsJt', M, W)
        M = M.reshape(D, M.shape[1] * d, M.shape[3] * d)
    return M[last_col]

def heisenberg_dense(L, J=1.0, pbc=False):
    """The same Hamiltonian built directly, term by term."""
    H = np.zeros((2**L, 2**L), dtype=complex)
    bonds = [(i, i + 1) for i in range(L - 1)] + ([(L - 1, 0)] if pbc else [])
    for (i, j) in bonds:
        for S in (sx, sy, sz):
            ops = [id2] * L
            ops[i] = S; ops[j] = S
            term = ops[0]
            for o in ops[1:]:
                term = np.kron(term, o)
            H += J * term
    return H

L = 6
H_mpo = mpo_dense(heisenberg_mpo(), L)
H_dir = heisenberg_dense(L)
print("max |H_MPO - H_direct| =", np.abs(H_mpo - H_dir).max())
print("number of terms in H for L=100:", 3 * 99, " bond dimension D = 5")

max |H_MPO - H_direct| = 0.0
number of terms in H for L=100: 297  bond dimension D = 5


### The operator Schmidt rank

Cut the chain after site $\ell$.  Viewed as a vector in operator space,
$\hat H$ is a matrix with row index $(s_1..s_\ell, s'_1..s'_\ell)$ and column
index $(s_{\ell+1}..s_L, s'_{\ell+1}..s'_L)$; its rank is the smallest
bond dimension any MPO can have at that cut.  The automaton gives *a*
valid $D$; only the rank gives *the* smallest.

In [3]:
def operator_schmidt_ranks(H, L, d=2, tol=1e-10):
    ranks = []
    T = H.reshape([d] * (2 * L))           # indices s_1..s_L, s'_1..s'_L
    for l in range(1, L):
        # bring (s_1..s_l, s'_1..s'_l) to the front
        perm = list(range(l)) + list(range(L, L + l)) + list(range(l, L)) + list(range(L + l, 2 * L))
        M = T.transpose(perm).reshape(d**(2 * l), d**(2 * (L - l)))
        sv = np.linalg.svd(M, compute_uv=False)
        ranks.append(int((sv > tol * sv[0]).sum()))
    return ranks

print("Heisenberg, open  (L=6):", operator_schmidt_ranks(heisenberg_dense(6), 6))
print("Heisenberg, ring  (L=6):", operator_schmidt_ranks(heisenberg_dense(6, pbc=True), 6))

# the pairing model with d=4 (empty, up, down, pair) on each level
def pairing_dense(L, g=1.0, xi=1.0):
    d = 4
    n_up = np.diag([0, 1, 0, 1.]); n_dn = np.diag([0, 0, 1, 1.])
    Pp = np.zeros((4, 4)); Pp[3, 0] = 1.0      # P^+ = a_up^+ a_dn^+ : |0> -> |pair>
    Pm = Pp.T
    def site_op(op, p):
        ops = [np.eye(4)] * L; ops[p] = op
        out = ops[0]
        for o in ops[1:]:
            out = np.kron(out, o)
        return out
    H = np.zeros((d**L, d**L))
    for p in range(L):
        H += xi * p * site_op(n_up + n_dn, p)
    for p in range(L):
        for q in range(L):
            H += -0.5 * g * site_op(Pp, p) @ site_op(Pm, q)
    return H

print("pairing, open     (L=4):", operator_schmidt_ranks(pairing_dense(4), 4, d=4))

Heisenberg, open  (L=6): [4, 5, 5, 5, 4]
Heisenberg, ring  (L=6): [4, 8, 8, 8, 4]
pairing, open     (L=4): [4, 4, 4]


### The staircase is a gauge

$\boldsymbol{W}\to\boldsymbol{X}^{-1}\boldsymbol{W}\boldsymbol{X}$ on every
bond leaves the operator unchanged and fills every block.

In [4]:
rng = np.random.default_rng(7)
X = rng.normal(size=(5, 5))
Xinv = np.linalg.inv(X)
W = heisenberg_mpo()
Wg = np.einsum('ab,bcst,cd->adst', Xinv, W, X)
# the boundary vectors transform with the gauge too:
def mpo_dense_bv(W, L, vl, vr):
    D, _, d, _ = W.shape
    M = np.einsum('a,abst->bst', vl, W)
    for _ in range(L - 1):
        M = np.einsum('bIJ,bcst->cIsJt', M, W)
        M = M.reshape(D, M.shape[1] * d, M.shape[3] * d)
    return np.einsum('b,bIJ->IJ', vr, M)

vl = np.zeros(5); vl[0] = 1
vr = np.zeros(5); vr[4] = 1
H_ref = mpo_dense_bv(W, 6, vl, vr)
H_gauged = mpo_dense_bv(Wg, 6, vl @ X, Xinv @ vr)
nonzero_blocks = sum(np.abs(Wg[a, b]).max() > 1e-12 for a in range(5) for b in range(5))
print("nonzero blocks after a random gauge:", nonzero_blocks, "of 25")
print("max |Delta H| =", np.abs(H_gauged - H_ref).max())

nonzero blocks after a random gauge: 25 of 25
max |Delta H| = 1.7763568394002505e-15


## 2. The norm matrix $\boldsymbol{G}$ and why canonical form matters

Freeze every tensor but the two-site centre $\boldsymbol{\Theta}$ at the
middle bond.  The state is $\sum_\alpha\Theta_\alpha|\phi_\alpha\rangle$,
and $G_{\alpha\beta}=\langle\phi_\alpha|\phi_\beta\rangle$ is the tensor
product of the norm environments of the left and right blocks.  For a
random MPS the block states are far from orthonormal; after
canonicalisation $\boldsymbol{G}=\boldsymbol{1}$ to machine precision --
and nothing physical has changed.

In [5]:
def random_mps(L, chi, d=2, seed=1):
    rng = np.random.default_rng(seed)
    dims = [1] + [min(chi, d**i, d**(L - i)) for i in range(1, L)] + [1]
    return [rng.normal(size=(dims[i], d, dims[i + 1])) for i in range(L)]

def norm_envs(M):
    """Left norm environments E_i = sum A^dagger ... A up to site i (exclusive)."""
    L = len(M)
    Ls = [np.ones((1, 1))]
    for A in M:
        E = Ls[-1]
        Ls.append(np.einsum('ab,asc,bsd->cd', E, A.conj(), A))
    Rs = [np.ones((1, 1))]
    for B in reversed(M):
        E = Rs[-1]
        Rs.append(np.einsum('asc,bsd,cd->ab', B.conj(), B, E))
    Rs = Rs[::-1]
    return Ls, Rs

def two_site_G(M, i):
    Ls, Rs = norm_envs(M)
    d = M[i].shape[1]
    G = np.einsum('ab,st,uv,cd->asucbtvd', Ls[i], np.eye(d), np.eye(d), Rs[i + 2])
    n = Ls[i].shape[0] * d * d * Rs[i + 2].shape[0]
    return G.reshape(n, n)

def left_canonicalize(M):
    """QR sweep from the left; returns left-canonical tensors and the final scalar."""
    out = []
    carry = np.ones((1, 1))
    for A in M:
        chi_l, d, chi_r = A.shape
        T = np.einsum('ab,bsc->asc', carry, A).reshape(-1, chi_r)
        Q, R = np.linalg.qr(T)
        out.append(Q.reshape(-1, d, Q.shape[1]))
        carry = R
    return out, carry

def mixed_canonical(M, i):
    """Left-canonical up to site i, right-canonical from i+1, two-site centre at (i, i+1)."""
    L = len(M)
    A, carry = left_canonicalize(M[:i])
    rest = [np.einsum('ab,bsc->asc', carry, M[i])] + [m.copy() for m in M[i + 1:]]
    # right-canonicalise rest from the right down to site i+1
    B = []
    carry = np.ones((1, 1))
    for T in reversed(rest[1:]):
        chi_l, d, chi_r = T.shape
        Tm = np.einsum('asb,bc->asc', T, carry).reshape(chi_l, -1)
        Q, R = np.linalg.qr(Tm.T)           # LQ via QR of the transpose
        B.insert(0, Q.T.reshape(Q.shape[1], d, chi_r))
        carry = R.T
    centre = np.einsum('asb,bc->asc', rest[0], carry)
    return A + [centre] + B

def mps_dense(M):
    v = M[0]
    for A in M[1:]:
        v = np.einsum('isb,bta->ista', v, A).reshape(1, -1, A.shape[2])
    return v.reshape(-1)

L, chi = 6, 4
M = random_mps(L, chi)
G = two_site_G(M, 2)
print("random MPS, L=6, chi=4:  cond(G) = %.3e" % np.linalg.cond(G))
Mc = mixed_canonical(M, 2)
Gc = two_site_G(Mc, 2)
print("same state, canonicalised: cond(G) = %.3f,  max|G-1| = %.1e"
      % (np.linalg.cond(Gc), np.abs(Gc - np.eye(len(Gc))).max()))
print("state unchanged: max |psi - psi_c| = %.1e" % np.abs(mps_dense(M) - mps_dense(Mc)).max())

# nearly parallel block states: a gauge X with two almost equal columns.
# cond(G) grows like delta^{-2}: at delta = 1e-8 the matrix is numerically singular.
for delta in (1e-6, 1e-8):
    Xp = np.eye(chi); Xp[:, 1] = Xp[:, 0] + delta * np.eye(chi)[:, 1]
    Mp = [m.copy() for m in Mc]
    Mp[1] = np.einsum('asb,bc->asc', Mp[1], Xp)
    Mp[2] = np.einsum('ab,bsc->asc', np.linalg.inv(Xp), Mp[2])
    print("two block states agreeing to %.0e: cond(G) = %.3e,  state unchanged to %.1e"
          % (delta, np.linalg.cond(two_site_G(Mp, 2)), np.abs(mps_dense(Mp) - mps_dense(Mc)).max()))

random MPS, L=6, chi=4:  cond(G) = 1.766e+07
same state, canonicalised: cond(G) = 1.000,  max|G-1| = 2.3e-16
state unchanged: max |psi - psi_c| = 5.3e-15
two block states agreeing to 1e-06: cond(G) = 4.002e+12,  state unchanged to 3.3e-10
two block states agreeing to 1e-08: cond(G) = 6.260e+16,  state unchanged to 2.9e-08


## 3. A two-site DMRG in NumPy

The ingredients are exactly the ones of the slides: environments
$\mathcal{L},\mathcal{R}$ of shape $(\chi, D, \chi)$, the matrix-free action
of $\hat H_{\rm eff}$ on $\boldsymbol{\Theta}$ (four contractions, cost
$\chi^3 d^2 D$), a Lanczos solver, the SVD that splits
$\boldsymbol{\Theta}$ and moves the centre, and the discarded weight.

In [6]:
def lanczos_lowest(matvec, v0, m=20, tol=1e-10):
    """Lowest eigenpair of a Hermitian operator given by matvec, with full
    reorthogonalisation.  Restarted until the Ritz residual is below tol."""
    v = v0 / np.linalg.norm(v0)
    for _ in range(6):
        V = [v]; alpha = []; beta = []
        w = matvec(v)
        for k in range(m):
            a = np.vdot(V[k], w).real; alpha.append(a)
            w = w - a * V[k] - (beta[-1] * V[k - 1] if beta else 0)
            for u in V:                       # full reorthogonalisation
                w = w - np.vdot(u, w) * u
            b = np.linalg.norm(w)
            if b < 1e-14 or k == m - 1:
                break
            beta.append(b); V.append(w / b); w = matvec(V[-1])
        T = np.diag(alpha) + np.diag(beta, 1) + np.diag(beta, -1)
        e, u = np.linalg.eigh(T)
        v = sum(u[k, 0] * V[k] for k in range(len(V)))
        v /= np.linalg.norm(v)
        resid = np.linalg.norm(matvec(v) - e[0] * v)
        if resid < tol:
            break
    return e[0], v

class DMRG:
    def __init__(self, W, L, chi_max, seed=0):
        self.W, self.L, self.chi = W, L, chi_max
        self.D, self.d = W.shape[0], W.shape[2]
        rng = np.random.default_rng(seed)
        self.M = [rng.normal(size=(1, self.d, 1)) + 0j for _ in range(L)]   # product state start
        self.canon = ['R'] * L
        self._right_canonicalize_all()
        # environments: Ls[i] = left env of bond before site i; Rs[i] = right env after site i
        vl = np.zeros(self.D); vl[0] = 1
        vr = np.zeros(self.D); vr[-1] = 1
        self.Ls = [None] * (L + 1); self.Rs = [None] * (L + 1)
        self.Ls[0] = vl.reshape(1, self.D, 1)
        self.Rs[L] = vr.reshape(1, self.D, 1)
        for i in range(L - 1, 0, -1):
            self.Rs[i] = self._contract_right(self.Rs[i + 1], self.M[i])
        self.eps = 0.0
        self.sv_mid = None

    def _right_canonicalize_all(self):
        carry = np.ones((1, 1))
        for i in range(self.L - 1, -1, -1):
            T = np.einsum('asb,bc->asc', self.M[i], carry)
            chi_l, d, chi_r = T.shape
            Q, R = np.linalg.qr(T.reshape(chi_l, -1).T)
            self.M[i] = Q.T.reshape(Q.shape[1], d, chi_r)
            carry = R.T
        self.M[0] *= carry[0, 0]            # absorb the overall normalisation

    def _contract_left(self, Lenv, A):
        # L_{a b a'} A*_{a s c} W_{b b' s s'} A_{a' s' c'} -> L'_{c b' c'}
        T = np.einsum('abx,asc->bsxc', Lenv, A.conj(), optimize=True)
        T = np.einsum('bsxc,bdst->dtxc', T, self.W, optimize=True)
        return np.einsum('dtxc,xte->cde', T, A, optimize=True)

    def _contract_right(self, Renv, B):
        # R_{b b' b''} B*_{c s b} W_{d b' s s'} B_{c' s' b''} -> R'_{c d c'}
        T = np.einsum('csb,bxy->csxy', B.conj(), Renv, optimize=True)
        T = np.einsum('csxy,dxst->cdty', T, self.W, optimize=True)
        return np.einsum('cdty,zty->cdz', T, B, optimize=True)

    def _matvec(self, Lenv, Renv, shape):
        W = self.W
        def mv(v):
            Th = v.reshape(shape)                               # (a', s', t', b')
            # W[b, d, u, s]: u is the bra (output) index, s the ket (input) index
            T = np.einsum('abx,xstc->absct', Lenv, Th, optimize=True)      # a b s t c   (b = MPO left)
            T = np.einsum('absct,bdus->adutc', T, W, optimize=True)        # a d u t c
            T = np.einsum('adutc,dfyt->aufyc', T, W, optimize=True)        # a u f y c   (f = MPO right)
            T = np.einsum('aufyc,gfc->auyg', T, Renv, optimize=True)       # a u y g
            return T.reshape(-1)
        return mv

    def _step(self, i, going_right, lanczos_tol):
        A, B = self.M[i], self.M[i + 1]
        Th = np.einsum('asb,btc->astc', A, B)
        shape = Th.shape
        mv = self._matvec(self.Ls[i], self.Rs[i + 2], shape)
        E, v = lanczos_lowest(mv, Th.reshape(-1), tol=lanczos_tol)
        Th = v.reshape(shape[0] * shape[1], shape[2] * shape[3])
        U, S, Vh = np.linalg.svd(Th, full_matrices=False)
        keep = min(self.chi, (S > 1e-14).sum())
        disc = (S[keep:]**2).sum()
        self.eps = max(self.eps, disc)
        U, S, Vh = U[:, :keep], S[:keep], Vh[:keep]
        S /= np.linalg.norm(S)
        if i == self.L // 2 - 1:
            self.sv_mid = S.copy()
        if going_right:
            self.M[i] = U.reshape(shape[0], shape[1], keep)
            self.M[i + 1] = (np.diag(S) @ Vh).reshape(keep, shape[2], shape[3])
            self.Ls[i + 1] = self._contract_left(self.Ls[i], self.M[i])
            self.canon[i], self.canon[i + 1] = 'L', 'C'
        else:
            self.M[i] = (U @ np.diag(S)).reshape(shape[0], shape[1], keep)
            self.M[i + 1] = Vh.reshape(keep, shape[2], shape[3])
            self.Rs[i + 1] = self._contract_right(self.Rs[i + 2], self.M[i + 1])
            self.canon[i], self.canon[i + 1] = 'C', 'R'
        return E

    def sweep(self, lanczos_tol=1e-8, watch=False):
        self.eps = 0.0
        if watch:
            print("start:        ", ' '.join(self.canon))
        for i in range(self.L - 1):
            E = self._step(i, True, lanczos_tol)
            if watch:
                print("bond (%d,%d): " % (i + 1, i + 2), ' '.join(self.canon), "  E = %.10f" % E)
        for i in range(self.L - 2, -1, -1):
            E = self._step(i, False, lanczos_tol)
            if watch:
                print("bond (%d,%d): " % (i + 1, i + 2), ' '.join(self.canon), "  E = %.10f" % E)
        return E

    def run(self, max_sweeps=20, tol=1e-10, verbose=False):
        E_old = np.inf
        for n in range(1, max_sweeps + 1):
            # loose eigensolver in the early sweeps, tight near convergence
            E = self.sweep(lanczos_tol=1e-5 if n <= 2 else 1e-10)
            if verbose:
                print("sweep %2d  E = %.12f  eps = %.1e" % (n, E, self.eps))
            if abs(E - E_old) < tol:
                break
            E_old = E
        return E, n

    def entropy_mid(self):
        p = self.sv_mid**2
        p = p[p > 1e-30]
        return float(-(p * np.log(p)).sum())

### One sweep, watched

`L` = left-canonical, `C` = the centre, `R` = right-canonical.  The centre
marches one site per step; at the turn the last bond is solved twice with
identical environments.

In [7]:
W = heisenberg_mpo()
dm = DMRG(W, L=6, chi_max=8)
dm.sweep(watch=True)
E6, _ = dm.run()
print("L=6, chi=8 (exact for L=6): E = %.12f   exact = %.12f"
      % (E6, np.linalg.eigvalsh(heisenberg_dense(6)).min()))

start:         R R R R R R
bond (1,2):  L C R R R R   E = -0.9982013240
bond (2,3):  L L C R R R   E = -1.1873031249
bond (3,4):  L L L C R R   E = -1.6779999409
bond (4,5):  L L L L C R   E = -2.0106590670
bond (5,6):  L L L L L C   E = -2.3186572522
bond (5,6):  L L L L C R   E = -2.3186572522
bond (4,5):  L L L C R R   E = -2.3779944202
bond (3,4):  L L C R R R   E = -2.3889783897
bond (2,3):  L C R R R R   E = -2.4847427291
bond (1,2):  C R R R R R   E = -2.4847427291
L=6, chi=8 (exact for L=6): E = -2.493577133888   exact = -2.493577133888


### Heisenberg at $L=32$

The table of the slides.  Report $\varepsilon_\chi$ (the largest discarded
weight of the last sweep) together with $\chi$: the same $\chi$ means very
different things for different models.

In [8]:
L = 32
print(" chi   E_DMRG              eps_chi    S_{L/2}   sweeps  time")
results = []
for chi in (8, 16, 32, 64):
    t0 = time.time()
    dm = DMRG(W, L=L, chi_max=chi, seed=0)
    E, n = dm.run(tol=1e-11)
    dt = time.time() - t0
    results.append((chi, E, dm.eps, dm.entropy_mid()))
    print("%4d  %.12f   %.1e   %.4f   %3d    %.1f s" % (chi, E, dm.eps, dm.entropy_mid(), n, dt))

 chi   E_DMRG              eps_chi    S_{L/2}   sweeps  time


   8  -13.995308101464   4.0e-05   0.6952     7    1.7 s


  16  -13.997257888143   3.1e-06   0.7202     6    2.5 s


  32  -13.997315266846   1.2e-08   0.7215     6    5.7 s


  64  -13.997315618007   6.9e-12   0.7215     5    10.8 s


### Extrapolation in the discarded weight

$E(\chi)-E_0\approx\langle\Delta E\rangle\,\varepsilon_\chi$ with a
proportionality constant that is an *energy*.  A line through the two most
accurate runs lands very close to the $\chi=64$ value; the ratio
$\Delta E/\varepsilon$ is of order one for this gapless chain, where the
relevant excitation energies are small.

In [9]:
chis, Es, eps, Ss = map(np.array, zip(*results))
E_ref = Es[-1]
for k in range(len(chis) - 1):
    print("chi = %2d:  (E - E_64)/eps = %.2f" % (chis[k], (Es[k] - E_ref) / eps[k]))
a, b = np.polyfit(eps[1:3], Es[1:3], 1)
print("linear extrapolation through chi=16,32 to eps=0: E = %.10f  (chi=64: %.10f)" % (b, E_ref))

chi =  8:  (E - E_64)/eps = 49.80
chi = 16:  (E - E_64)/eps = 18.65
chi = 32:  (E - E_64)/eps = 28.27
linear extrapolation through chi=16,32 to eps=0: E = -13.9973154980  (chi=64: -13.9973156180)


## 4. Checks of the week-40 answers

### The stability matrix from Wick's theorem, against brute force

For a random two-body Hamiltonian in a small space we build the Hartree-Fock
determinant, assemble $\Delta+A$ and $B$ from the formulas of the exercise,
and compare $\frac12\langle\chi|\hat M|\chi\rangle$ with the second
derivative of the exact $E(\delta C)$ along the manifold, computed in the
full configuration space with the bit representation of week 38.

In [10]:
from itertools import combinations

def random_hamiltonian(n, seed=3):
    rng = np.random.default_rng(seed)
    h = rng.normal(size=(n, n)); h = (h + h.T) / 2
    v = rng.normal(size=(n, n, n, n)) * 0.3
    v = v + v.transpose(1, 0, 3, 2)                       # <pq|v|rs> = <qp|v|sr>
    v = v + v.transpose(2, 3, 0, 1)                       # hermitian (real)
    v = v - v.transpose(1, 0, 2, 3)                       # antisymmetrise: <pq|v|rs>_AS
    return h, v

def hartree_fock(h, v, N, iters=200):
    n = len(h)
    C = np.eye(n)
    for _ in range(iters):
        rho = C[:, :N] @ C[:, :N].T
        f = h + np.einsum('prqs,rs->pq', v, rho)
        e, C = np.linalg.eigh(f)
    return e, C, f

def transform(h, v, C):
    h2 = C.T @ h @ C
    v2 = np.einsum('pa,qb,rc,sd,pqrs->abcd', C, C, C, C, v)
    return h2, v2

def fci_hamiltonian(h, v, N):
    n = len(h)
    dets = [sum(1 << p for p in c) for c in combinations(range(n), N)]
    idx = {d: k for k, d in enumerate(dets)}
    def apply(det, ops):
        """ops: list of (p, dagger); apply right to left, return (sign, det) or None."""
        sign = 1
        for p, dag in reversed(ops):
            occ = (det >> p) & 1
            if dag == occ:
                return None
            sign *= (-1) ** bin(det & ((1 << p) - 1)).count('1')
            det ^= (1 << p)
        return sign, det
    H = np.zeros((len(dets), len(dets)))
    for k, d in enumerate(dets):
        for p in range(n):
            for q in range(n):
                r = apply(d, [(p, 1), (q, 0)])
                if r: H[idx[r[1]], k] += r[0] * h[p, q]
        for p in range(n):
            for q in range(n):
                for r_ in range(n):
                    for s in range(n):
                        r = apply(d, [(p, 1), (q, 1), (s, 0), (r_, 0)])
                        if r: H[idx[r[1]], k] += 0.25 * r[0] * v[p, q, r_, s]
    return H, dets, idx

n, N = 6, 3
h, v = random_hamiltonian(n)
eps_hf, C, f = hartree_fock(h, v, N)
hH, vH = transform(h, v, C)                       # Hartree-Fock basis
Hfci, dets, idx = fci_hamiltonian(hH, vH, N)
ref = sum(1 << i for i in range(N)); k0 = idx[ref]
E0 = Hfci[k0, k0]
holes, parts = range(N), range(N, n)
pairs = [(a, i) for a in parts for i in holes]
print("Brillouin: max |f_ai| in the HF basis =", np.abs((C.T @ f @ C)[N:, :N]).max())

# Delta + A and B from the Wick formulas of the exercise
Dp = np.zeros((len(pairs), len(pairs))); B = np.zeros_like(Dp)
for x, (a, i) in enumerate(pairs):
    for y, (b, j) in enumerate(pairs):
        Dp[x, y] = (eps_hf[a] - eps_hf[i]) * (a == b) * (i == j) + vH[a, j, i, b]
        B[x, y] = vH[a, b, i, j]
Mst = np.block([[Dp, B], [B, Dp]])

# brute force: |c'> = exp(T)|c> in the FCI space, E(dC) exactly, second derivative numerically
def excite(det, a, i):
    occ_i, occ_a = (det >> i) & 1, (det >> a) & 1
    if not occ_i or occ_a: return None
    sign = (-1) ** bin(det & ((1 << i) - 1)).count('1'); det ^= (1 << i)
    sign *= (-1) ** bin(det & ((1 << a) - 1)).count('1'); det ^= (1 << a)
    return sign, det

def Tmat(dC):
    T = np.zeros(Hfci.shape, dtype=complex)
    for x, (a, i) in enumerate(pairs):
        for k, d in enumerate(dets):
            r = excite(d, a, i)
            if r: T[idx[r[1]], k] += dC[x] * r[0]
    return T

def energy(dC):
    T = Tmat(dC)
    expT = np.eye(len(dets)) + T + T @ T / 2 + T @ T @ T / 6   # T^4 = 0 for N=3
    psi = expT[:, k0]
    return (psi.conj() @ Hfci @ psi) / (psi.conj() @ psi)

rng = np.random.default_rng(11)
u = rng.normal(size=len(pairs))
hstep = 1e-4
def second_derivative(direction):
    # E(t) = E_0 + t^2/2 <chi|M|chi> + O(t^3), chi = (dC, dC*), dC = t*direction
    return (energy(hstep * direction) - 2 * energy(0 * direction) + energy(-hstep * direction)).real / hstep**2
print("E(0) - E_HF = %.1e" % (energy(0 * u) - E0).real)
for label, dC, chi in (("real dC:      ", u, np.concatenate([u, u])),
                       ("imaginary dC: ", 1j * u, np.concatenate([1j * u, -1j * u]))):
    print(label, "Wick <chi|M|chi> = %.8f   brute-force d^2E/dt^2 = %.8f"
          % ((chi.conj() @ Mst @ chi).real, second_derivative(dC)))
print("The real direction feels (Delta+A)+B, the imaginary one (Delta+A)-B: both blocks are checked.")
print("lowest eigenvalue of M: %.6f  (the Hartree-Fock determinant of this Hamiltonian is stable)"
      % np.linalg.eigvalsh(Mst).min())

Brillouin: max |f_ai| in the HF basis = 2.0014705901886258e-15
E(0) - E_HF = 0.0e+00
real dC:       Wick <chi|M|chi> = 44.42944144   brute-force d^2E/dt^2 = 44.42943968
imaginary dC:  Wick <chi|M|chi> = 46.20578539   brute-force d^2E/dt^2 = 46.20578409
The real direction feels (Delta+A)+B, the imaginary one (Delta+A)-B: both blocks are checked.
lowest eigenvalue of M: 1.925074  (the Hartree-Fock determinant of this Hamiltonian is stable)


### Positive semidefinite: Lipkin with $N=2$, and the pairing block

In [11]:
def lipkin_M(eps, V):
    return np.array([[eps, 0, 0, V], [0, eps, V, 0], [0, V, eps, 0], [V, 0, 0, eps]])

def principal_minors(M, size):
    n = len(M)
    return [np.linalg.det(M[np.ix_(S, S)]) for S in combinations(range(n), size)]

eps = 1.0
print("   V   diag>=0  2x2 minors>=0  3x3 minors>=0  det>=0  trace>=0  lambda_min")
for V in (0.5, 0.9, 1.0, 1.1, 1.5):
    M = lipkin_M(eps, V)
    print("%5.2f    %5s      %5s          %5s        %5s    %5s     %+.3f" % (
        V, bool(np.all(np.diag(M) >= 0)),
        bool(min(principal_minors(M, 2)) >= -1e-12),
        bool(min(principal_minors(M, 3)) >= -1e-12),
        bool(np.linalg.det(M) >= -1e-12),
        bool(np.trace(M) >= 0),
        np.linalg.eigvalsh(M).min()))
print("  det M = (eps^2 - V^2)^2 never goes negative; the 2x2 minors eps^2 - V^2 fail first, at |V| = eps.")

print()
for g in (0.5, 1.0, 2.0, 3.0):
    Delta = 1.0 + g / 2                      # eps_q - eps_p = 1
    blk = np.array([[Delta, -g / 2], [-g / 2, Delta]])
    print("pairing block, g = %.1f: minor = %.3f, eigenvalues = %s  (eps_q - eps_p = 1, +g = %.1f)"
          % (g, np.linalg.det(blk), np.linalg.eigvalsh(blk), 1 + g))

   V   diag>=0  2x2 minors>=0  3x3 minors>=0  det>=0  trace>=0  lambda_min
 0.50     True       True           True         True     True     +0.500
 0.90     True       True           True         True     True     +0.100
 1.00     True       True           True         True     True     +0.000
 1.10     True      False          False         True     True     -0.100
 1.50     True      False          False         True     True     -0.500
  det M = (eps^2 - V^2)^2 never goes negative; the 2x2 minors eps^2 - V^2 fail first, at |V| = eps.

pairing block, g = 0.5: minor = 1.500, eigenvalues = [1.  1.5]  (eps_q - eps_p = 1, +g = 1.5)
pairing block, g = 1.0: minor = 2.000, eigenvalues = [1. 2.]  (eps_q - eps_p = 1, +g = 2.0)
pairing block, g = 2.0: minor = 3.000, eigenvalues = [1. 3.]  (eps_q - eps_p = 1, +g = 3.0)
pairing block, g = 3.0: minor = 4.000, eigenvalues = [1. 4.]  (eps_q - eps_p = 1, +g = 4.0)
